In [7]:
!pip install langchain-huggingface
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from groq import Groq
class PDFChatbot:
    def __init__(self, pdf_path, groq_api_key):
        self.history = []
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50
        )
        docs = splitter.split_documents(documents)
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        self.vectorstore = FAISS.from_documents(docs, embeddings)
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})
        self.client = Groq(api_key=groq_api_key)
    def ask(self, question):
        self.history.append({
            "role": "user",
            "content": question
        })
        retrieved_docs = self.retriever.invoke(question)
        context = "\n\n".join(
            [doc.page_content for doc in retrieved_docs]
        )
        history_text = ""
        for msg in self.history:
            history_text += f"{msg['role']}: {msg['content']}\n"
        prompt = f"""
You are a helpful PDF chatbot.
Conversation History:
{history_text}
PDF Context:
{context}
User Question:
{question}
Answer clearly using the PDF context and previous conversation.
"""
        response = self.client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )
        answer = response.choices[0].message.content
        self.history.append({
            "role": "assistant",
            "content": answer
        })
        return answer
    def show_history(self):
        print("\n FULL CHAT HISTORY")
        print("=" * 50)
        for i, msg in enumerate(self.history, start=1):
            print(f"\n{i}. {msg['role'].upper()}:")
            print(msg['content'])
pdf_path = r"C:\Users\hp\Downloads\DeepLearning_Notes.pdf"
groq_api_key = "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"
bot = PDFChatbot(pdf_path, groq_api_key)
questions = [
    "What is the main topic of the PDF?",
    "Can you explain that in simpler terms?",
    "What are the important concepts mentioned?",
    "How are those concepts connected?",
    "Give me a short final summary of everything discussed."
]
for q in questions:
    print(f"\n USER: {q}")
    answer = bot.ask(q)
    print(f"\n BOT: {answer}")
    print("-" * 60)
bot.show_history()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


 USER: What is the main topic of the PDF?

 BOT: The main topic of the PDF appears to be "AI / ML INTERNSHIP PROGRAM - WEEK 3 — STUDENT NOTES" specifically focusing on "Deep Learning Foundations & Generative AI".
------------------------------------------------------------

 USER: Can you explain that in simpler terms?

 BOT: I'd be happy to explain the PDF's main topic in simpler terms.

The PDF is about a course on Artificial Intelligence (AI) and Machine Learning (ML), specifically focusing on the basics of Deep Learning. Deep Learning is a type of AI that helps computers learn complex patterns in data.

Imagine you're trying to predict if an email is spam or not. The PDF shows us how a computer can do this by using a special math formula that involves "weights" and "biases." Think of weights like the importance of different factors (e.g., the number of "free" words in the email), and biases like a small adjustment that helps the computer make its prediction.

The process of learni